In [3]:
# ============================================================
# INGEST CONSTRUCTORS — LOCAL EXECUTION
# ============================================================

import os
from pyspark.sql.types import StructType, StructField, StringType

In [4]:
# -----------------------------------------
# 1. IMPORT ENVIRONMENT CONFIG
# -----------------------------------------
try:
    BRONZE_LANDING
except NameError:
    import nbformat
    from nbconvert import PythonExporter

    project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
    config_path = os.path.join(project_root, "/Users/manoharazalki/F1-Analytics/notebooks/00-common/01_environment_config.ipynb")
    with open(config_path, "r") as f:
        nb = nbformat.read(f, as_version=4)

    exporter = PythonExporter()
    source, _ = exporter.from_notebook_node(nb)
    exec(source)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/08 23:07:53 WARN Utils: Your hostname, Manohars-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 10.68.78.178 instead (on interface en0)
26/06/08 23:07:53 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/08 23:07:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/08 23:07:54 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/06/08 23:07:54 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/06/08 23:07:54 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


✔ Spark Session Initialized
✔ Environment Config Loaded
PROJECT_ROOT: /Users/manoharazalki/F1-ANALYTICS


In [ ]:
# -----------------------------------------
# 2. IMPORT HELPER FUNCTIONS
# -----------------------------------------
try:
    add_ingestion_metadata
except NameError:
    import nbformat
    from nbconvert import PythonExporter

    helper_path = "/Users/manoharazalki/F1-Analytics/notebooks/00-common/02_bronze_helpers.ipynb"
    with open(helper_path, "r") as f:
        nb = nbformat.read(f, as_version=4)

    exporter = PythonExporter()
    source, _ = exporter.from_notebook_node(nb)
    exec(source)

✔ Bronze helper functions loaded


In [6]:
# -----------------------------------------
# 3. Define source + target paths
# -----------------------------------------
source_file = f"{BRONZE_LANDING}/constructors.json"
target_folder = f"{BRONZE_ROOT}/constructors"
target_file = f"{target_folder}/constructors.parquet"

os.makedirs(target_folder, exist_ok=True)

print("Source:", source_file)
print("Target:", target_file)

Source: /Users/manoharazalki/F1-ANALYTICS/data/bronze/landing/constructors.json
Target: /Users/manoharazalki/F1-ANALYTICS/data/bronze/constructors/constructors.parquet


In [7]:
# -----------------------------------------
# 4. Define schema (same as Databricks)
# -----------------------------------------
constructors_schema = StructType([
    StructField("constructorId", StringType(), True),
    StructField("name", StringType(), True),
    StructField("nationality", StringType(), True),
    StructField("url", StringType(), True)
])

In [8]:
# -----------------------------------------
# 5. Read JSON
# -----------------------------------------
constructors_df = (
    spark.read
        .format("json")
        .schema(constructors_schema)
        .option("mode", "FAILFAST")
        .load(source_file)
)

print("✔ Constructors file read successfully")
constructors_df.show(5, truncate=False)

✔ Constructors file read successfully
+-------------+----------+-----------+-----------------------------------------------------------------+
|constructorId|name      |nationality|url                                                              |
+-------------+----------+-----------+-----------------------------------------------------------------+
|adams        |Adams     |american   |http://en.wikipedia.org/wiki/Adams_(constructor)                 |
|afm          |AFM       |german     |http://en.wikipedia.org/wiki/Alex_von_Falkenhausen_Motorenbau    |
|ags          |AGS       |french     |http://en.wikipedia.org/wiki/Automobiles_Gonfaronnaises_Sportives|
|alfa         |Alfa Romeo|swiss      |http://en.wikipedia.org/wiki/Alfa_Romeo_in_Formula_One           |
|alphatauri   |AlphaTauri|italian    |http://en.wikipedia.org/wiki/Scuderia_AlphaTauri                 |
+-------------+----------+-----------+-----------------------------------------------------------------+
only showing top 

In [9]:
# -----------------------------------------
# 6. Add metadata
# -----------------------------------------
constructors_final_df = add_ingestion_metadata(constructors_df, source_file)

print("✔ Metadata added")
constructors_final_df.show(5, truncate=False)

✔ Metadata added
+-------------+----------+-----------+-----------------------------------------------------------------+--------------------------+-----------------------------------------------------------------------+
|constructorId|name      |nationality|url                                                              |ingest_timestamp          |source_file                                                            |
+-------------+----------+-----------+-----------------------------------------------------------------+--------------------------+-----------------------------------------------------------------------+
|adams        |Adams     |american   |http://en.wikipedia.org/wiki/Adams_(constructor)                 |2026-06-08 23:07:56.463504|/Users/manoharazalki/F1-ANALYTICS/data/bronze/landing/constructors.json|
|afm          |AFM       |german     |http://en.wikipedia.org/wiki/Alex_von_Falkenhausen_Motorenbau    |2026-06-08 23:07:56.463504|/Users/manoharazalki/F1-ANALYTICS/da

In [10]:
# -----------------------------------------
# 7. Write to Bronze (Option C)
# -----------------------------------------
(
    constructors_final_df
        .write
        .mode("overwrite")
        .parquet(target_file)
)

print(f"✔ Constructors Bronze written to: {target_file}")

✔ Constructors Bronze written to: /Users/manoharazalki/F1-ANALYTICS/data/bronze/constructors/constructors.parquet


In [11]:
# -----------------------------------------
# 8. Read back for validation
# -----------------------------------------
spark.read.parquet(target_file).show(5, truncate=False)

+-------------+----------+-----------+-----------------------------------------------------------------+--------------------------+-----------------------------------------------------------------------+
|constructorId|name      |nationality|url                                                              |ingest_timestamp          |source_file                                                            |
+-------------+----------+-----------+-----------------------------------------------------------------+--------------------------+-----------------------------------------------------------------------+
|adams        |Adams     |american   |http://en.wikipedia.org/wiki/Adams_(constructor)                 |2026-06-08 23:07:56.539546|/Users/manoharazalki/F1-ANALYTICS/data/bronze/landing/constructors.json|
|afm          |AFM       |german     |http://en.wikipedia.org/wiki/Alex_von_Falkenhausen_Motorenbau    |2026-06-08 23:07:56.539546|/Users/manoharazalki/F1-ANALYTICS/data/bronze/landing